Before we begin, let us execute the below cell to display information about the NVIDIA® CUDA® driver and the GPUs running on the server by running the `nvidia-smi` command. To do this, execute the cell block below by clicking on it with your mouse, and pressing Ctrl+Enter, or pressing the play button in the toolbar above. You should see some output returned below the grey cell.

In [ ]:
nvidia-smi

# Learning objectives
The **goal** of this lab is to:

- Apply parallelisation techniques to loops using standard keywords in C++ and Fortran across both CPU and GPU architectures.
- Examine methods for safely managing shared variables by using atomic operations to prevent race conditions.
- Utilise the concept and benefits of unified memory for efficient data sharing between CPU and GPU.
- Enhance parallel performance by implementing techniques such as loop collapsing.
- Select appropriate compiler flags to compile parallel Fortran code for both CPU and GPU targets.

We do not intend to cover:
- Detailed optimization techniques and mapping of standard constructs to CUDA Fortran

**NOTE**: To be able to see the Nsight Systems profiler output, please download the latest version of Nsight Systems from [here](https://developer.nvidia.com/nsight-systems).

# Fortran Standard Parallelism

ISO Standard Fortran 2008 introduced the DO CONCURRENT construct to allow you to express loop-level parallelism, one of the various mechanisms for expressing parallelism directly in the Fortran language. 

Fortran developers have  been able to accelerate their programs using CUDA Fortran, OpenACC or OpenMP. Now with the support of DO CONCURRENT on GPU with NVIDIA HPC SDK, the compiler automatically accelerates loops using DO CONCURRENT, allowing developers to get the benefit of accelerating  on NVIDIA GPUs using ISO Standard Fortran without any extensions, directives, or non-standard libraries. You can now write standard Fortran, remaining fully portable to other compilers and systems, and still benefit from the full power of NVIDIA GPUs

For our code to make *Pair Calculation* all that’s required is expressing loops with DO CONCURRENT. The example below will introduce you to the syntax of DO CONCURRENT 

Sample vector addition code is shown in code below:

```fortran
subroutine vec_addition(x,y,n)
  real :: x(:), y(:)
  integer :: n, i  
  do i = 1, n 
    y(i) = x(i)+y(i)
  end do  
end subroutine vec_addition
```

In order to make use of ISO Fortran DO CONCURRENT we need to replace the `do` loop with `do concurrent` as shown in code below

```fortran
subroutine vec_addition(x,y,n)
  real :: x(:), y(:)
  integer :: n, i  
  do concurrent (i = 1: n) 
      y(i) = x(i)+y(i)
  end do  
end subroutine vec_addition
```

By changing the DO loop to DO CONCURRENT, you are telling the compiler that there are no data dependencies between the n loop iterations. This leaves the compiler free to generate instructions that the iterations can be executed in any order and simultaneously. The compiler parallelizes the loop even if there are data dependencies, resulting in race conditions and likely incorrect results. It’s your responsibility to ensure that the loop is safe to be parallelized.

### Nested Loop Parallelism

Nested loops are a common code pattern encountered in HPC applications. A simple example might look like the following:

```fortran
do i = 1, n
  do j = 1, m
    a(i, j) = c(i) + b(j) 
  end do
end do
```

It is straightforward to write such patterns with a single DO CONCURRENT statement, as in the following example. It is easier to read, and the compiler has more information available for optimization.

```fortran
do concurrent(i = 1:n, j = 1:m)
  a(i, j) = c(i) + b(j)
end do
```


# Standard Exercise

Lets start modifying the original code and add the necessary changes to parallelise the code. Without changing the orginal code, you will get error running the below cells.

**Click on the <b>[**source code**](../source_code/rdf.f90)</b> link, and start modifying the RDF code.**

To help you modify the code, some sections are marked with `TODO: ` comments consisting of simple instructions.  Where additional modifications from the [original source code](../../_common/source_code/rdf.f90) were required they are marked with `Note: ` comments for you to review. Remember to **SAVE** your code after changes, before running the below cells.

## Compile and Run for Multicore

Now, let's compile the code. We will be using NVIDIA HPC SDK for this exercise. The flags used for enabling standard parallelism for target offloading are as follows:

- `-stdpar` : This flag enables standard parallelism for the target architecture
- `-stdpar=multicore` will allow us to compile our code for a multicore
- `-stdpar` will allow us to compile our code for a NVIDIA GPU (Default is NVIDIA)

After running the cells, you can inspect part of the compiler feedback and see what it's telling us (your compiler feedback will be similar to the below).

### Compile the code for multicore

In [ ]:
#Compile the code for multicore
cd ../source_code && printf "Compiling for multicore ...\n" && make clean && make rdf_f && 
printf "\nRunning the executable and validating the output\n" && ./rdf_f && cat Pair_entropy.dat

**Note:** Since we are targeting the NVTX v3 API, a header-only C library, and added Fortran-callable wrappers to the code, we add `-lnvhpcwrapnvtx` at the compile time to do the link to the library.

The output should be the following:

```
s2 value is -2.43191
s2bond value is -3.87015
    
```

and an example compiler feedback would look similar as below:
    
```
pair_gpu:
    175, Generating Multicore code
        175, Loop parallelized across CPU threads
```

In [ ]:
#profile and see output of nvptx
cd ../source_code && nsys profile -t nvtx --stats=true --force-overwrite true -o rdf_doconcurrent_multicore ./rdf_f

Let's checkout the profiler's report. Download and save the report file by holding down the Shift key and right-clicking the [report link](../source_code/rdf_doconcurrent_multicore.nsys-rep) then choosing Save Link As. Once done, open it via the GUI. From the _Timeline View_, checkout the NVTX markers displays as part of threads. **Why are we using NVTX?** Please see the Moodle section on [Using NVIDIA Tools Extension (NVTX)](https://129.234.196.13/moodle/mod/page/view.php?id=29).

From the _Timeline View_, right click on the nvtx row and click the "show in events view". Now you can see the nvtx statistic at the bottom of the window which shows the duration of each range.

**Example screenshot (multicore)**
    
<img src="../../_common/images/do_concurrent_multicore.png">


## Compile and run for NVIDIA GPU

Without changing the code now let us try to recompile the code for NVIDIA GPU and rerun. GPU acceleration of standard parallel algorithms is enabled with the `-⁠stdpar` command-line option when using NVIDIA HPC Fortran compiler. If `-⁠stdpar `is specified, almost all algorithms that use a parallel execution policy are compiled for offloading to run in parallel on an NVIDIA GPU.

**Understand and analyze** the [solution](../source_code/SOLUTION/rdf.f90) and compare with your version. Once done, compile your code by running below cells.

Make sure to validate the output by running the executable and validate the output.

### Compile for Tesla GPU

In [ ]:
#compile for Tesla GPU
cd ../source_code && printf "Compiling for GPU ...\n"  && nvfortran -stdpar=gpu -Minfo=stdpar -acc -o rdf_f rdf.f90 -lnvhpcwrapnvtx &&
printf "\nRunning the executable and validating the output\n" && ./rdf_f && cat Pair_entropy.dat

**Note:** Since we are targeting the NVTX v3 API, a header-only C library, and added Fortran-callable wrappers to the code, we add `-lnvhpcwrapnvtx` at the compile time to do the link to the library.

The output should be the following:

```
s2 value is -2.43191
s2bond value is -3.87015
    
```

and an example compiler feedback would look similar as below:
    
```
pair_gpu:
    175, Generating implicit private(ind,r,dz,dy,dx)
         Generating NVIDIA GPU code
        175,   ! blockidx%x threadidx%x auto-collapsed
             Loop parallelized across CUDA thread blocks, CUDA threads(128) collapse(2) ! blockidx%x threadidx%x
    175, Generating implicit copyin(d_x(:)) [if not already present]
         Generating implicit copy(d_g(:)) [if not already present]
         Generating implicit copyin(d_y(:),d_z(:)) [if not already present]
```


In [ ]:
#profile and see output of nvptx
cd ../source_code && nsys profile -t nvtx,cuda --stats=true --force-overwrite true -o rdf_doconcurrent_gpu ./rdf_f

Let's checkout the profiler's report. Download and save the report file by holding down the Shift key and right-clicking the [report link](../source_code/rdf_doconcurrent_gpu.nsys-rep) then choosing Save Link As. Once done, open it via the GUI.

From the "_Timeline View_" on the top pane, double click on the "CUDA" from the function table on the left and expand it. Hover your mouse over the CUDA row (underlined with blue color in the below screenshot) and expand it till you see both kernels and memory row.  Zoom in on the timeline and you can see a pattern similar to the screenshot below. The blue boxes (annotated with a red box) are the compute kernels. The small red and green boxes (annotated with a green box) represent data movements.  Similarly, expanding the "Threads" reveals a "CUDA API" (annoted with a purple box) which shows the different CUDA API used. Notice the CudaMallocManaged API indicating the use of Unified (managed) memory.

**Example screenshot (GPU)**
    
<img src="../../_common/images/do_concurrent_gpu.png">


If you inspect the output of the profiler closer, you can see the *Unified Memory* usage. Moreover, if you compare the NVTX marker `Pair_Calculation` (from the NVTX row) in both multicore and GPU version, you can see how much improvement you achieved. 

Feel free to checkout the [solution](../source_code/SOLUTION/rdf.f90) again to help you understand better.

# Standard Language Analysis

## Usage Scenarios
DO CONCURRENT is part of the Fortran standard language and provides a good start for accelerating code on accelerators like GPU and multicores.

## Limitations/Constraints
This isn’t a catch-all solution or necessarily an alternative to other programming models that provide more control over things like thread management. *DO CONCURRENT* provides the highest portability and can be seen as the first step to porting on GPU. The general abstraction limits the optimization functionalities. For example, the implementations are currently dependent on Unified memory. Moreover, one does not have control over thread management and that will limit performance improvement.

## Which Compilers Support stdpar on GPUs and Multicore?
1. NVIDIA GPU: As of Jan 2021, the HPC SDK compiler from NVIDIA supports std::par and DO-CONCURRENT on NVIDIA GPU.
2. x86 Multicore: DO CONCURRENT: Other compilers like intel compiler have an implementation on a multicore CPU

# Saving the exercise

If you would like to download this exercise for later viewing, it is recommended you go to your browser's file menu (not the Jupyter notebook file menu) and save the complete web page.  This will ensure the images are copied down as well. You can also execute the following cell block to create a zip file of the files you have been working on, and download it with the link below.

In [ ]:
cd ..
rm -f _files.zip
zip -r _files.zip *

**After** executing the above zip command, you should be able to download and save the zip file by holding down Shift and right-clicking [Here](../_files.zip) then choosing save Link As.

# Links and Resources
[Blog post on Developing Accelerated Code with Standard Language Parallelism](https://developer.nvidia.com/blog/developing-accelerated-code-with-standard-language-parallelism/)

[Blog post on Accelerating Fortran DO CONCURRENT with GPUs and the NVIDIA HPC SDK](https://developer.nvidia.com/blog/accelerating-fortran-do-concurrent-with-gpus-and-the-nvidia-hpc-sdk/)

[NVIDIA Nsight System](https://docs.nvidia.com/nsight-systems/)

# Licensing 

Copyright © 2022 OpenACC-Standard.org.  This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.